<a href="https://colab.research.google.com/github/thaoly-corner/reddit-iphone-feedback-pipeline/blob/dev/notebooks/DE_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [ ]:
%cd /content
!git clone -b dev https://github.com/thaoly-corner/reddit-iphone-feedback-pipeline.git
%cd /content/reddit-iphone-feedback-pipeline
!pip install -q -r requirements.txt

/content
Cloning into 'reddit-iphone-feedback-pipeline'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 155 (delta 36), reused 151 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 98.42 KiB | 806.00 KiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/reddit-iphone-feedback-pipeline
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%cd /content/reddit-iphone-feedback-pipeline

/content/reddit-iphone-feedback-pipeline


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Access key loaded:", bool(os.getenv("AWS_ACCESS_KEY_ID")))
print("Secret key loaded:", bool(os.getenv("AWS_SECRET_ACCESS_KEY")))
print("Region:", os.getenv("AWS_DEFAULT_REGION"))

Access key loaded: True
Secret key loaded: True
Region: ap-southeast-1


In [ ]:
import sys
sys.path.append('/content/reddit-iphone-feedback-pipeline')

from dotenv import load_dotenv
load_dotenv()

from config.pipeline_config import StorageConfig
from processing.batch.bronze_to_silver.spark_cleaner import build_spark_session

storage = StorageConfig()

spark = build_spark_session(
    "Reddit_iPhone_VoC",
    storage
)

bronze = spark.read.parquet(storage.bronze_posts_path)

print("Bronze rows:", bronze.count())
bronze.printSchema()

Bronze rows: 13959
root
 |-- _meta: string (nullable = true)
 |-- all_awardings: string (nullable = true)
 |-- allow_live_comments: boolean (nullable = true)
 |-- approved_at_utc: double (nullable = true)
 |-- approved_by: double (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_flair_background_color: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_richtext: string (nullable = true)
 |-- author_flair_template_id: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_flair_text_color: string (nullable = true)
 |-- author_flair_type: string (nullable = true)
 |-- author_fullname: string (nullable = true)
 |-- author_is_blocked: boolean (nullable = true)
 |-- author_patreon_flair: double (nullable = true)
 |-- author_premium: double (nullable = true)
 |-- awarders: string (nullable = true)
 |-- banned_at_utc: double (nullable = true)
 |-- banned_by: 

In [ ]:
test_path = f"s3a://{storage.s3_bucket}/silver/_iam_test"

(
    bronze
    .limit(2)
    .write
    .mode("overwrite")
    .parquet(test_path)
)

print("WRITE OK")

WRITE OK


In [ ]:
!python -m scripts.run_pipeline \
  --stage silver \
  --silver-engine pyspark_roberta

Active PipelineExecutionConfig: PipelineExecutionConfig(silver_engine='pyspark_roberta', batch_gold_engine='spark_sql', stream_engine='apache_flink', serving_backend='postgres', schema_paradigm='galaxy', max_acceptable_p99_latency_ms=50.0, max_acceptable_memory_mb=512.0)
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-p

In [ ]:
silver_df = spark.read.parquet(storage.silver_posts_path)

print("Silver rows:", silver_df.count())

silver_df.groupBy("sentiment").count().show()

silver_df.groupBy(
    "feedback_year",
    "feedback_month"
).count().orderBy(
    "feedback_year",
    "feedback_month"
).show()

Silver rows: 13812
+---------+-----+
|sentiment|count|
+---------+-----+
| POSITIVE| 1314|
| NEGATIVE| 6126|
|  NEUTRAL| 6372|
+---------+-----+

+-------------+--------------+-----+
|feedback_year|feedback_month|count|
+-------------+--------------+-----+
|         2026|             5| 5043|
|         2026|             6| 4606|
|         2026|             7| 4163|
+-------------+--------------+-----+



In [ ]:
!python -m scripts.run_pipeline \
  --stage gold \
  --batch-gold-engine spark_sql \
  --stream-engine apache_flink

Active PipelineExecutionConfig: PipelineExecutionConfig(silver_engine='pyspark_roberta', batch_gold_engine='spark_sql', stream_engine='apache_flink', serving_backend='postgres', schema_paradigm='galaxy', max_acceptable_p99_latency_ms=50.0, max_acceptable_memory_mb=512.0)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-p

In [ ]:
gold_marts = [
    "feedback_volume",
    "issue_summary",
    "emerging_issues",
    "sentiment_summary",
    "feature_feedback",
    "high_impact_complaints",
    "issue_trends",
    "model_comparison",
    "recurring_issues",
    "support_demand",
]

for name in gold_marts:
    df = spark.read.parquet(f"{storage.gold_base_path}/{name}")
    print(name, df.count())

feedback_volume 92
issue_summary 4272
emerging_issues 2056
sentiment_summary 3138
feature_feedback 12
high_impact_complaints 500
issue_trends 162
model_comparison 771
recurring_issues 33
support_demand 1185


In [ ]:
!python -m scripts.run_pipeline \
  --stage serving \
  --serving-backend postgres \
  --schema-paradigm galaxy

Active PipelineExecutionConfig: PipelineExecutionConfig(silver_engine='pyspark_roberta', batch_gold_engine='spark_sql', stream_engine='apache_flink', serving_backend='postgres', schema_paradigm='galaxy', max_acceptable_p99_latency_ms=50.0, max_acceptable_memory_mb=512.0)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
:: loading settings :: url = jar:file:/usr/local/lib/python3.13/dist-p